# AIPW simulation (legacy two-factor params)

Uses the usual joint two-factor law (`loading=0.8`, `φ=0.3`, `cov(X,Z)=0.64`),
mean 2, propensity `logit(π) = -0.6 + Z`, correct specification, main `aipw`.

This notebook runs **n = 250**, **R = 1000**.  
`RunConfig.n` also accepts a list (e.g. `[250, 500, 1000]`); each `n` gets its own `runs/sim_K_n…` folder.

**Note:** augmented sandwich uses finite differences — R=1000 will take a long time.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "simulation.py").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "legacy"))
sys.path.insert(0, str(ROOT))

from dgp_two_factor_helpers import joint_sigma_from_two_factor
from metrics import summarize_run
from simulation import RunConfig, run_simulation

In [2]:
# Legacy-style DGP knobs (same as prior misspec / mest runs)
p = 6
Sigma = joint_sigma_from_two_factor(
    p=p,
    loading=0.8,
    phi_offdiag=0.3,
    cov_XZ=0.64,
    var_Z=1.0,
)
mean = [2.0] * (p + 1)

# Sample sizes: list supported, e.g. n = [250, 500, 1000]
n = [250]
reps = 1000
seed = 20261123

config = RunConfig(
    n=n,
    reps=reps,
    Sigma=Sigma,
    mean=mean,
    n_z=1,
    seed=seed,
    scenario="correct",
    methods=("aipw",),
    propensity_intercept=-0.6,
    propensity_coef=1.0,
    use_true_pi=False,
    save_data=False,
)
config

RunConfig(n=[250], reps=1000, Sigma=array([[1.   , 0.64 , 0.64 , 0.192, 0.192, 0.192, 0.64 ],
       [0.64 , 1.   , 0.64 , 0.192, 0.192, 0.192, 0.64 ],
       [0.64 , 0.64 , 1.   , 0.192, 0.192, 0.192, 0.64 ],
       [0.192, 0.192, 0.192, 1.   , 0.64 , 0.64 , 0.64 ],
       [0.192, 0.192, 0.192, 0.64 , 1.   , 0.64 , 0.64 ],
       [0.192, 0.192, 0.192, 0.64 , 0.64 , 1.   , 0.64 ],
       [0.64 , 0.64 , 0.64 , 0.64 , 0.64 , 0.64 , 1.   ]]), mean=[2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0], n_z=1, seed=20261123, run_id=None, run_dir=None, scenario='correct', outcome_z='true', propensity_z='true', methods=('aipw',), propensity_intercept=-0.6, propensity_coef=1.0, save_data=False, use_true_pi=False)

In [3]:
# Writes under runs/sim_K (or sim_K_n250 if several n). Never overwrites.
run_dirs = run_simulation(config)
if not isinstance(run_dirs, list):
    run_dirs = [run_dirs]

for d in run_dirs:
    print(d)

KeyboardInterrupt: 

## Summary

`run_simulation` already writes `SUMMARY.json` / `summary_*.csv`.  
Re-run the summarizer below if you want to refresh after the fact.

In [ ]:
summaries = []
for run_dir in run_dirs:
    summary = summarize_run(run_dir, write=True)
    overall = summary["overall_table"].copy()
    overall.insert(0, "run_id", run_dir.name)
    overall.insert(1, "run_dir", str(run_dir))
    summaries.append(overall)
    print(run_dir.name)
    display(overall)

summary_table = pd.concat(summaries, ignore_index=True)
summary_table

In [ ]:
# Per-entry covariance metrics (bias, emp var, M-est var_ratio, coverage)
entry_tables = []
for run_dir in run_dirs:
    path = run_dir / "output" / "summary_aipw_covariance_by_entry.csv"
    if path.exists():
        df = pd.read_csv(path)
        df.insert(0, "run_id", run_dir.name)
        entry_tables.append(df)

entry = pd.concat(entry_tables, ignore_index=True) if entry_tables else None
entry

In [ ]:
# CFA rates / parameter summary when present
for run_dir in run_dirs:
    cfa_path = run_dir / "output" / "summary_aipw_cfa_by_param.csv"
    rates = pd.read_csv(run_dir / "output" / "summary_overall.csv")
    print(run_dir.name, "rates")
    display(rates[["method", "psd_rate", "heywood_rate", "cfa_converged_rate",
                   "cov_mean_var_ratio", "cov_mean_coverage95", "cov_rel_bias"]])
    if cfa_path.exists():
        print(run_dir.name, "CFA params")
        display(pd.read_csv(cfa_path))